# Notebook 6a: Does fine-tuning learn the job, or memorise the resume?

**Before running:** Runtime → Change runtime type → **T4 GPU**. Close any other GPU session first (Runtime → Manage sessions → Terminate).

**Run time:** about 2–2.5 hours (15 fine-tuning runs). **Resumable:** every finished model's test scores are saved to Drive. If Colab disconnects, just **Run all** again and it skips what is already done.

**Why this notebook:** in Notebook 3 the fine-tuned BGE scored NDCG@10 0.833, but only 0.791 when the test resumes were removed from training. That was one split with 45 test jobs, too few to be sure. Notebook 3b then showed that the Good/Potential label follows the *resume*, not the job. So we suspect fine-tuning partly memorises "this resume is usually Good".

**Design: a 5 × 5 grid, fold k tests on held-out jobs AND held-out resumes.**
Jobs are split into 5 groups and resumes into 5 groups. In fold k, the **test pairs** are those whose job is in job-group k and whose resume is in resume-group k. The same test pairs are used for three fine-tuned models:

| Model | Trained on | Has it seen the test resumes? |
|---|---|---|
| **standard** | every pair whose job is not in group k | **yes**, with other jobs |
| **strict** | pairs whose job is not in group k **and** resume is not in group k | no |
| **size-matched** | a random subset of *standard*, the same size as *strict* | yes |

- *standard − strict* = the total advantage of having seen the resumes.
- *size-matched − strict* = the same advantage with **equal training size**. If this stays large, the gain comes from **resume memorisation**, not from more data.

Reference methods on the same test pairs (no training): TF-IDF, zero-shot BGE (the model before fine-tuning), zero-shot E5, and the **resume-prior** built from each model's own training labels.

Fine-tuning settings are **identical to Notebook 3** (BGE-base, CoSENT, 2 epochs, lr 2e-5, batch 16, 10% warm-up, fp16, seed 42) and were fixed in advance.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!pip -q install -U sentence-transformers

import os, time, gc
import numpy as np
import pandas as pd
import torch
from sklearn.metrics import roc_auc_score

assert torch.cuda.is_available(), 'No GPU: Runtime > Change runtime type > T4 GPU, then run again'
device = 'cuda'
DATA = '/content/drive/MyDrive/resume-job-matching/data'
RESULTS = '/content/drive/MyDrive/resume-job-matching/results'
CACHE = f'{RESULTS}/cv_cache'
os.makedirs(CACHE, exist_ok=True)
SEED = 42; K = 5

all_df = pd.concat([pd.read_parquet(f'{DATA}/{s}.parquet') for s in ('train', 'val', 'test')], ignore_index=True)
all_df['pair_key'] = all_df.job_id + '|' + all_df.resume_id

rng = np.random.default_rng(SEED)
jobs = all_df.job_id.unique(); res = all_df.resume_id.unique()
job_fold = dict(zip(jobs, rng.permutation(np.arange(len(jobs)) % K)))
res_fold = dict(zip(res, rng.permutation(np.arange(len(res)) % K)))
all_df['jf'] = all_df.job_id.map(job_fold); all_df['rf'] = all_df.resume_id.map(res_fold)

def fold_sets(k):
    test = all_df[(all_df.jf == k) & (all_df.rf == k)]
    standard = all_df[all_df.jf != k]
    strict = all_df[(all_df.jf != k) & (all_df.rf != k)]
    matched = standard.sample(n=len(strict), random_state=SEED + k)
    return test, {'standard': standard, 'strict': strict, 'size-matched': matched}

design = []
for k in range(K):
    test, trains = fold_sets(k)
    g = test.groupby('job_id').relevance
    design.append({'fold': k, 'test pairs': len(test), 'test jobs': test.job_id.nunique(),
                   'jobs with 2+ labels': int((g.nunique() >= 2).sum()),
                   'test labels': test.label.value_counts().to_dict(),
                   **{f'train {n}': len(d) for n, d in trains.items()},
                   'size-matched pairs with a test resume': int(trains['size-matched'].resume_id.isin(test.resume_id).sum())})
display(pd.DataFrame(design))

## 1. Metrics
The test cell of each fold has only a few candidates per job, so per-job NDCG@10 would be unstable. The main measures are **pair-level AUC**, pooled over all 5 folds (about 1,600 pairs):
- **AUC fit-vs-no**: Good or Potential vs No Fit
- **AUC good-vs-potential**: Good vs Potential, among the fit pairs (the label Notebook 3b showed to be resume-driven)

We also report **NDCG@5** per job (jobs with at least 3 candidates and 2 label levels) and **mean per-job AUC**. Confidence intervals and the paired tests use a **bootstrap over test jobs**, so the pairs from one job stay together.

In [ ]:
def dcg(rels):
    rels = np.asarray(rels, dtype=float)
    return np.sum((2 ** rels - 1) / np.log2(np.arange(2, len(rels) + 2)))
def ndcg_at_k(r, k):
    ideal = dcg(sorted(r, reverse=True)[:k]); return dcg(r[:k]) / ideal if ideal > 0 else 0.0
def auc(y, s):
    y = np.asarray(y); return roc_auc_score(y, s) if 0 < y.sum() < len(y) else np.nan

def pooled(df, col):
    fit = df[df.relevance > 0]
    return {'AUC fit-vs-no': auc((df.relevance > 0).astype(int), df[col]),
            'AUC good-vs-potential': auc((fit.relevance == 2).astype(int), fit[col])}

def per_job(df, col):
    r = np.random.default_rng(SEED); rows = []
    for jid, g in df.groupby('job_id'):
        rel = g.relevance.to_numpy(); s = g[col].to_numpy(); fit = rel > 0
        row = {'job_id': jid}
        if len(g) >= 3 and len(set(rel)) >= 2:
            row['NDCG@5'] = ndcg_at_k(rel[np.lexsort((r.random(len(s)), -s))].tolist(), 5)
        row['job AUC fit-vs-no'] = auc(fit.astype(int), s)
        row['job AUC good-vs-potential'] = auc((rel[fit] == 2).astype(int), s[fit]) if fit.sum() >= 2 else np.nan
        rows.append(row)
    return pd.DataFrame(rows).set_index('job_id')

def job_bootstrap(df, fn, n=1000):
    """fn(df) -> float. Resample whole jobs."""
    r = np.random.default_rng(SEED); idx = list(df.reset_index(drop=True).groupby('job_id').indices.values())
    d0 = df.reset_index(drop=True)
    vals = [fn(d0.iloc[np.concatenate([idx[i] for i in r.integers(0, len(idx), len(idx))])]) for _ in range(n)]
    return np.nanpercentile(vals, [2.5, 97.5])

## 2. Fine-tuning (same function as Notebook 3) and fold loop
Each model's scores on its fold's test pairs are saved to `results/cv_cache/` before the next one starts.

In [ ]:
from datasets import Dataset
from sentence_transformers import SentenceTransformer, SentenceTransformerTrainer, SentenceTransformerTrainingArguments, losses

def finetune(pairs, tag, base='BAAI/bge-base-en-v1.5'):
    model = SentenceTransformer(base, device=device); model.max_seq_length = 512
    ds = Dataset.from_dict({'sentence1': pairs.job_text.tolist(), 'sentence2': pairs.resume_text.tolist(),
                            'score': (pairs.relevance / 2).astype(float).tolist()}).shuffle(seed=SEED)
    args = SentenceTransformerTrainingArguments(
        output_dir=f'/content/ft-{tag}', num_train_epochs=2, per_device_train_batch_size=16,
        learning_rate=2e-5, warmup_ratio=0.1, fp16=True, seed=SEED,
        save_strategy='no', logging_steps=100, report_to='none')
    SentenceTransformerTrainer(model=model, args=args, train_dataset=ds, loss=losses.CoSENTLoss(model)).train()
    return model

def cosine_scores(model, test, qp='', dp=''):
    je = dict(zip(test.job_id.unique(), model.encode([qp + t for t in test.drop_duplicates('job_id').job_text],
                                                     normalize_embeddings=True, batch_size=32)))
    rd = test.drop_duplicates('resume_id')
    re_ = dict(zip(rd.resume_id, model.encode([dp + t for t in rd.resume_text], normalize_embeddings=True, batch_size=32)))
    return np.array([float(je[j] @ re_[r]) for j, r in zip(test.job_id, test.resume_id)])

def free(m):
    del m; gc.collect(); torch.cuda.empty_cache()

for k in range(K):
    test, trains = fold_sets(k)
    for name, train in trains.items():
        path = f'{CACHE}/fold{k}_{name}.parquet'
        if os.path.exists(path):
            print(f'fold {k} {name}: cached'); continue
        t0 = time.perf_counter()
        model = finetune(train, f'{k}-{name}')
        out = test[['pair_key']].assign(score=cosine_scores(model, test))
        # resume-prior from THIS model's training labels: what memorising the resume alone would give
        tr = train.groupby('resume_id').relevance.mean()
        out['prior'] = test.resume_id.map(tr).fillna(train.relevance.mean()).to_numpy()
        out.to_parquet(path, index=False)
        free(model)
        print(f'fold {k} {name}: {len(train)} train pairs, {(time.perf_counter() - t0) / 60:.1f} min')

## 3. Reference methods on the same test pairs (no training on labels)

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

ref_path = f'{CACHE}/reference.parquet'
if os.path.exists(ref_path):
    ref = pd.read_parquet(ref_path)
else:
    test_all = pd.concat([fold_sets(k)[0] for k in range(K)], ignore_index=True)
    tfidf = TfidfVectorizer(stop_words='english', ngram_range=(1, 2), min_df=2, sublinear_tf=True, max_features=100_000)
    tfidf.fit(list(all_df.drop_duplicates('job_id').job_text) + list(all_df.drop_duplicates('resume_id').resume_text))
    ref = test_all[['pair_key']].copy()
    ref['TF-IDF'] = np.asarray(tfidf.transform(test_all.job_text).multiply(tfidf.transform(test_all.resume_text)).sum(axis=1)).ravel()
    for name, hf_id, qp, dp in [('BGE-base (zero-shot)', 'BAAI/bge-base-en-v1.5', '', ''),
                                ('E5-base (zero-shot)', 'intfloat/e5-base-v2', 'query: ', 'passage: ')]:
        m = SentenceTransformer(hf_id, device=device); m.max_seq_length = 512
        ref[name] = cosine_scores(m, test_all, qp, dp); free(m)
    ref.to_parquet(ref_path, index=False)

## 4. Results (all 5 folds pooled)

In [ ]:
test_all = pd.concat([fold_sets(k)[0].assign(fold=k) for k in range(K)], ignore_index=True)
ev = test_all[['pair_key', 'job_id', 'resume_id', 'fold', 'relevance', 'label']].merge(ref, on='pair_key').reset_index(drop=True)
for name in ['standard', 'strict', 'size-matched']:
    s = pd.concat([pd.read_parquet(f'{CACHE}/fold{k}_{name}.parquet') for k in range(K)])
    ev = ev.merge(s.rename(columns={'score': f'FT {name}', 'prior': f'Resume-prior ({name} train)'}), on='pair_key').reset_index(drop=True)

METHODS = ['FT standard', 'FT size-matched', 'FT strict', 'BGE-base (zero-shot)', 'E5-base (zero-shot)', 'TF-IDF',
           'Resume-prior (standard train)', 'Resume-prior (strict train)']
rows = []
for m in METHODS:
    p = pooled(ev, m); pj = per_job(ev, m)
    row = {'method': m}
    for metric, v in p.items():
        lo, hi = job_bootstrap(ev, lambda d, m=m, metric=metric: pooled(d, m)[metric])
        row[metric] = round(v, 3); row[metric + ' CI'] = f'[{lo:.3f}, {hi:.3f}]'
    row.update({'NDCG@5': round(pj['NDCG@5'].mean(), 3), 'NDCG@5 jobs': int(pj['NDCG@5'].notna().sum()),
                'mean job AUC good-vs-potential': round(pj['job AUC good-vs-potential'].mean(), 3)})
    rows.append(row)
summary = pd.DataFrame(rows)
print(f'{len(ev)} test pairs, {ev.job_id.nunique()} jobs, {ev.resume_id.nunique()} resumes over {K} folds')
display(summary)

## 5. Paired tests: is the advantage of seeing the resumes real?
Bootstrap over test jobs (1,000 resamples), two-sided. Positive difference = the first model is better.

In [ ]:
def paired(a, b, metric, n=2000):
    r = np.random.default_rng(SEED); idx = list(ev.groupby('job_id').indices.values())
    f = lambda d: pooled(d, a)[metric] - pooled(d, b)[metric]
    base = f(ev); boots = np.array([f(ev.iloc[np.concatenate([idx[i] for i in r.integers(0, len(idx), len(idx))])]) for _ in range(n)])
    boots = boots[~np.isnan(boots)]; lo, hi = np.percentile(boots, [2.5, 97.5])
    return {'comparison': f'{a}  −  {b}', 'metric': metric, 'diff': round(base, 3), 'CI': f'[{lo:.3f}, {hi:.3f}]',
            'p': round(min(1, 2 * min((boots <= 0).mean(), (boots >= 0).mean())), 4)}

tests = pd.DataFrame([paired(a, b, metric)
                      for a, b in [('FT standard', 'FT strict'), ('FT size-matched', 'FT strict'),
                                   ('FT strict', 'BGE-base (zero-shot)'), ('FT strict', 'TF-IDF')]
                      for metric in ['AUC fit-vs-no', 'AUC good-vs-potential']])
display(tests)

### How to read this
- **size-matched − strict clearly positive on good-vs-potential** → the fine-tuned model gains from having seen the resumes, not from more data. That is memorisation, and it inflates standard benchmark scores.
- **Resume-prior (standard train)** shows how far memorising the resume alone gets you, with no job text at all. If *FT standard* is close to it on good-vs-potential, that part of its score is the resume, not the match.
- **FT strict vs zero-shot BGE** says whether fine-tuning helps at all once memorisation is ruled out.

## 6. Per-fold view (a stability check)

In [ ]:
fold_rows = []
for k in range(K):
    d = ev[ev.fold == k]
    for m in ['FT standard', 'FT size-matched', 'FT strict', 'BGE-base (zero-shot)', 'Resume-prior (standard train)']:
        fold_rows.append({'fold': k, 'method': m, **{x: round(v, 3) for x, v in pooled(d, m).items()}})
display(pd.DataFrame(fold_rows).pivot(index='method', columns='fold', values='AUC good-vs-potential'))

## 7. Save

In [ ]:
pd.DataFrame(design).to_csv(f'{RESULTS}/06a_cv_design.csv', index=False)
summary.to_csv(f'{RESULTS}/06a_cv_summary.csv', index=False)
tests.to_csv(f'{RESULTS}/06a_cv_tests.csv', index=False)
pd.DataFrame(fold_rows).to_csv(f'{RESULTS}/06a_cv_per_fold.csv', index=False)
ev.to_parquet(f'{RESULTS}/06a_cv_pair_scores.parquet', index=False)
print('saved')